# 02 - Data.gov and data.gov mirror parsing

data.gov mirror from: https://source.coop/harvard-lil/gov-data

In this notebook I parse json files for data.gov (dgf) and data.gov mirror (dgm), (exact and fuzzy) match entries from Data Rescue Project (DRP) to either data catalogue, analyse the differences between DRP vs dgm vs dgf, and (as I decided to use dgf due to more metadata availability and small description differences) find corresponding entries from DRP in dgf.

<p align="center">
  <img src="00_figures/drp_preprocessing_02_01.png"
       alt="DRP preprocessing pipeline"
       title="DRP Preprocessing pipeline, part covered in the notebook in red."
       height="600">
</p>


In [1]:
ENV = "local"

from pathlib import Path

if ENV == "colab":
    DATA_DIR = Path("/content/drive/MyDrive/thesis")
else:
    DATA_DIR = Path("00_data")

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')


## Imports

In [2]:
import json
import re
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import orjson
import pandas as pd
from pandas import json_normalize
import matplotlib.pyplot as plt


import pyarrow as pa
import pyarrow.parquet as pq

from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer


from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    classification_report, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score,
    cross_validate, train_test_split,
)

from xgboost import XGBClassifier

from collections import defaultdict
import torch


In [5]:
import pandas as pd
from pandas import json_normalize
import json
import pyarrow as pa
import pyarrow.parquet as pq
import time
import orjson
import re
from rapidfuzz import fuzz, process
import numpy as np
import pandas as pd
from collections import Counter
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import orjson
import re
import time
from pathlib import Path

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [6]:
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from general.util.banned_words import add_flagged_column_vectorized

In [7]:
def to_str(v):
    if isinstance(v, str): return v
    if v is None: return ''
    return json.dumps(v)

def to_list(v):
    return v if isinstance(v, list) else []

def safe_join(items, key):
    try:
        if not items or not isinstance(items, list):
            return None
        return ','.join(str(item.get(key, '')) for item in items if isinstance(item, dict) and item.get(key))
    except Exception:
        return None

## Data

### data.gov

In [ ]:
writer = None
batch = []
batch_size = 100_000
total = 0
errors = 0
skipped_empty = 0
start = time.time()

with open(str(DATA_DIR / 'datagov_full.jsonl'), 'rb') as f:
    for line in f:
        try:
            r = orjson.loads(line)
            total += 1

            title = r.get('title', '') or ''
            description = r.get('description', '') or ''

            if not title and not description:
                skipped_empty += 1
                continue

            org = r.get('organization') or {}
            dcat = r.get('dcat') or {}

            batch.append({
                'title': title,
                'description': description,
                'publisher': r.get('publisher', '') or '',
                'identifier': r.get('identifier', '') or '',
                'slug': r.get('slug', '') or '',
                'has_spatial': r.get('has_spatial'),
                'popularity': r.get('popularity'),
                'last_harvested_date': r.get('last_harvested_date', '') or '',
                'keyword': to_list(r.get('keyword')),
                'theme': to_list(r.get('theme')),
                'org_name': org.get('name', '') or '',
                'org_type': org.get('organization_type', '') or '',
                'org_id': org.get('id', '') or '',
                'org_slug': org.get('slug', '') or '',
                'org_description': to_str(org.get('description')),
                'dcat_issued': to_str(dcat.get('issued')),
                'dcat_modified': to_str(dcat.get('modified')),
                'dcat_access_level': to_str(dcat.get('accessLevel')),
                'dcat_access_comment': to_str(dcat.get('accessLevelComment')),
                'dcat_landing_page': to_str(dcat.get('landingPage')),
                'dcat_language': to_list(dcat.get('language')),
                'dcat_temporal': to_str(dcat.get('temporal')),
                'dcat_spatial': to_str(dcat.get('spatial')),
                'dcat_bureau_code': to_list(dcat.get('bureauCode')),
                'dcat_program_code': to_list(dcat.get('programCode')),
                'dcat_periodicity': to_str(dcat.get('accrualPeriodicity')),
                'dcat_license': to_str(dcat.get('license')),
                'dcat_rights': to_str(dcat.get('rights')),
            })

            if total % 100_000 == 0:
                elapsed = time.time() - start
                rate = total / elapsed
                print(f"  Parsed {total:,} rows | {rate:,.0f} rows/sec | {errors} errors")

            if len(batch) >= batch_size:
                table = pa.Table.from_pylist(batch)
                if writer is None:
                    writer = pq.ParquetWriter(str(DATA_DIR / '02_dgf.parquet'), table.schema)
                    print(f"  Schema: {table.schema.names}")
                writer.write_table(table)
                print(f"  Wrote batch to parquet ({total:,} total)")
                batch = []

        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  ERROR row {total}: {type(e).__name__}: {e}")

if batch:
    table = pa.Table.from_pylist(batch)
    if writer is None:
        writer = pq.ParquetWriter(str(DATA_DIR / '02_dgf.parquet'), table.schema)
    writer.write_table(table)

if writer:
    writer.close()

elapsed = time.time() - start
print(f"\nDone: {total:,} rows in {elapsed:.1f}s ({total/elapsed:,.0f} rows/sec)")
print(f"Skipped (empty): {skipped_empty}")
print(f"Errors: {errors}")
print(f"Output: 02_dgf.parquet")

  Parsed 100,000 rows | 30,910 rows/sec | 0 errors
  Schema: ['title', 'description', 'publisher', 'identifier', 'slug', 'has_spatial', 'popularity', 'last_harvested_date', 'keyword', 'theme', 'org_name', 'org_type', 'org_id', 'org_slug', 'org_description', 'dcat_issued', 'dcat_modified', 'dcat_access_level', 'dcat_access_comment', 'dcat_landing_page', 'dcat_language', 'dcat_temporal', 'dcat_spatial', 'dcat_bureau_code', 'dcat_program_code', 'dcat_periodicity', 'dcat_license', 'dcat_rights']
  Wrote batch to parquet (100,000 total)
  Parsed 200,000 rows | 24,333 rows/sec | 0 errors
  Wrote batch to parquet (200,000 total)
  Parsed 300,000 rows | 23,831 rows/sec | 0 errors
  Wrote batch to parquet (300,000 total)
  Parsed 400,000 rows | 23,025 rows/sec | 0 errors
  Wrote batch to parquet (400,000 total)
  Parsed 500,000 rows | 20,840 rows/sec | 0 errors
  Wrote batch to parquet (500,000 total)

Done: 514,352 rows in 26.5s (19,382 rows/sec)
Skipped (empty): 0
Errors: 0
Output: 02_dgf.par

In [ ]:
dgf = pd.read_parquet(str(DATA_DIR / '02_dgf.parquet'))
print(f"dgf_full shape: {dgf.shape}")

### data.gov mirror

In [12]:
writer = None
batch = []
batch_size = 100_000
total = 0
errors = 0
start = time.time()

with open(str(DATA_DIR / 'metadata.jsonl'), 'rb') as f:
    for line in f:
        try:
            r = orjson.loads(line)
            sm = r.get('signed_metadata') or {}
            if isinstance(sm, str):
                try:
                    sm = orjson.loads(sm)
                except Exception:
                    sm = {}
            if not isinstance(sm, dict):
                sm = {}

            dgm = sm.get('data_gov_metadata') or {}
            if not isinstance(dgm, dict):
                dgm = {}
            org = dgm.get('organization') or {}
            if not isinstance(org, dict):
                org = {}

            batch.append({
                'id': sm.get('id'),
                'url': sm.get('url'),
                'description': sm.get('description'),

                'dgm_id': dgm.get('id'),
                'dgm_name': dgm.get('name'),
                'dgm_title': dgm.get('title'),
                'dgm_notes': dgm.get('notes'),
                'dgm_state': dgm.get('state'),
                'dgm_type': dgm.get('type'),
                'dgm_url': dgm.get('url'),
                'dgm_version': dgm.get('version'),
                'dgm_author': dgm.get('author'),
                'dgm_maintainer': dgm.get('maintainer'),
                'dgm_metadata_created': dgm.get('metadata_created'),
                'dgm_metadata_modified': dgm.get('metadata_modified'),
                'dgm_num_resources': dgm.get('num_resources'),
                'dgm_num_tags': dgm.get('num_tags'),
                'dgm_owner_org': dgm.get('owner_org'),
                'dgm_private': dgm.get('private'),
                'dgm_isopen': dgm.get('isopen'),
                'dgm_license_title': dgm.get('license_title'),

                'org_id': org.get('id'),
                'org_name': org.get('name'),
                'org_title': org.get('title'),
                'org_type': org.get('type'),
                'org_description': org.get('description'),
                'org_created': org.get('created'),
                'org_is_organization': org.get('is_organization'),
                'org_approval_status': org.get('approval_status'),
                'org_state': org.get('state'),
                'tags': safe_join(dgm.get('tags'), 'name'),
            })
            total += 1

            if total % 100_000 == 0:
                elapsed = time.time() - start
                rate = total / elapsed
                print(f"  Parsed {total:,} rows | {rate:,.0f} rows/sec | {errors} errors")

            if len(batch) >= batch_size:
                table = pa.Table.from_pylist(batch)
                if writer is None:
                    writer = pq.ParquetWriter(str(DATA_DIR / '02_dgm.parquet'), table.schema)
                    print(f"  Schema: {table.schema.names}")
                writer.write_table(table)
                print(f"  Wrote batch to parquet ({total:,} total)")
                batch = []

        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"  ERROR row {total}: {type(e).__name__}: {e}")

if batch:
    table = pa.Table.from_pylist(batch)
    if writer is None:
        writer = pq.ParquetWriter(str(DATA_DIR / '02_dgm.parquet'), table.schema)
    writer.write_table(table)

if writer:
    writer.close()

elapsed = time.time() - start
print(f"\nDone: {total:,} rows in {elapsed:.1f}s ({total/elapsed:,.0f} rows/sec)")
print(f"Errors: {errors}")
print(f"Output: 02_dgm.parquet")

  Parsed 100,000 rows | 14,782 rows/sec | 0 errors
  Schema: ['id', 'url', 'description', 'dgm_id', 'dgm_name', 'dgm_title', 'dgm_notes', 'dgm_state', 'dgm_type', 'dgm_url', 'dgm_version', 'dgm_author', 'dgm_maintainer', 'dgm_metadata_created', 'dgm_metadata_modified', 'dgm_num_resources', 'dgm_num_tags', 'dgm_owner_org', 'dgm_private', 'dgm_isopen', 'dgm_license_title', 'org_id', 'org_name', 'org_title', 'org_type', 'org_description', 'org_created', 'org_is_organization', 'org_approval_status', 'org_state', 'tags']
  Wrote batch to parquet (100,000 total)
  Parsed 200,000 rows | 10,855 rows/sec | 0 errors
  Wrote batch to parquet (200,000 total)
  Parsed 300,000 rows | 9,888 rows/sec | 0 errors
  Wrote batch to parquet (300,000 total)

Done: 311,820 rows in 35.5s (8,778 rows/sec)
Errors: 0
Output: 02_dgm.parquet


,id,url,description,dgm_id,dgm_name,dgm_title,dgm_notes,dgm_state,dgm_type,dgm_url,...,org_id,org_name,org_title,org_type,org_description,org_created,org_is_organization,org_approval_status,org_state,tags
0,1ce2d0a1-8c2d-47a2-89cb-b68122ff0099,https://catalog.data.gov/dataset/afsc-race-sap...,"Archive of data.gov dataset ""AFSC/RACE/SAP/Arm...",36a5bb7a-c948-40ee-9272-61e7833f7041,afsc-race-sap-armistead-1975-2016-eastern-beri...,AFSC/RACE/SAP/Armistead: 1975 - 2016 eastern B...,The Resource Assessment and Conservation Engin...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"alaska,alaska fisheries science center,bering ..."
1,0d24350b-ae44-49d1-9ebd-8fd5d5b3ed4f,https://catalog.data.gov/dataset/tiger-line-sh...,"Archive of data.gov dataset ""TIGER/Line Shapef...",4106992e-d05e-49d1-ab5f-21c9d109d8b2,tiger-line-shapefile-2023-county-moca-municipi...,"TIGER/Line Shapefile, 2023, County, Moca Munic...",The TIGER/Line shapefiles and related database...,active,dataset,None,...,fb3131aa-ef06-4a00-ad84-67d93a71d7e3,census-gov,"U.S. Census Bureau, Department of Commerce",organization,The Census Bureau's mission is to serve as the...,2020-11-10T14:08:17.917195,True,approved,active,"72099,area hydrography identifier,county fips ..."
2,c464f1c5-b1e7-4252-8674-686ce9d46c84,https://catalog.data.gov/dataset/fbsab-recruit...,"Archive of data.gov dataset ""FBSAB RECRUIT Ree...",28782773-af58-4116-b5df-ee584f819ccb,fbsab-recruit-reef-fish-belt-transect-survey-a...,FBSAB RECRUIT Reef Fish Belt Transect Survey a...,Shore-based belt transects were conducted at 1...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"10367,belt transect survey,biology,central pac..."
3,bd3fb449-c524-4e6f-85a4-86fac4f1107b,https://catalog.data.gov/dataset/yellowknife-n...,"Archive of data.gov dataset ""Yellowknife, N. W...",5520ddd6-d0f6-463d-bd4d-4a734dbda691,yellowknife-n-w-t-nt-cyzf4,"Yellowknife, N. W. T., NT (CYZF)","Timeseries data from 'Yellowknife, N. W. T., N...",active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"aggregate_quality_flag,air_pressure_at_mean_se..."
4,ac27c8d9-33a2-4552-886a-c8b4e7418af1,https://catalog.data.gov/dataset/l02194-nos-hy...,"Archive of data.gov dataset ""L02194: NOS Hydro...",4410ddef-9a08-4b8c-ba12-03c190a54123,l02194-nos-hydrographic-survey,L02194: NOS Hydrographic Survey,The National Oceanic and Atmospheric Administr...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"bathymetry,bathymetry/seafloor topography,cont..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311815,38ac7f7f-f3a5-4680-940e-314919c38ac2,https://catalog.data.gov/dataset/ow-noaa-avhrr...,"Archive of data.gov dataset ""OW NOAA AVHRR-GAC...",a3e6e94a-c3f9-467a-8e42-82e34428c0b4,ow-noaa-avhrr-gac-sea-surface-temperature1,OW NOAA AVHRR-GAC Sea-Surface Temperature,The dataset contains satellite-derived sea-sur...,active,dataset,None,...,5f4f1195-e770-4a2a-8f75-195cd98860ce,noaa-gov,National Oceanic and Atmospheric Administratio...,organization,,2020-11-10T15:36:13.098184,True,approved,active,"3-day,avhrr,daily,doc/noaa/nmfs/pifsc,gac,glob..."
311816,4d52f844-35e4-4391-bf50-0920db67f3c9,https://catalog.data.gov/dataset/mora-county-b...,"Archive of data.gov dataset ""Mora County Block...",74567044-a1d7-421c-81e2-76bcd7576058,mora-county-blocks-age-by-5-year-age-groups-fo...,"Mora County Blocks, Age by 5-Year Age Groups f...",The once-a-decade decennial census was conduct...,active,dataset,None,...,6a3f

In [ ]:
dgm = pd.read_parquet('00_data/02_dgm.parquet')
dgm

### DRP

As created in notebook 01_data_prep_drp

In [3]:
drp = pd.read_csv(str(DATA_DIR / 'drp_withdoi.csv'))

In [4]:
drp.shape

(2539, 107)

# Matching

Normalization

In [16]:
US_STATES = {
    'alabama','alaska','arizona','arkansas','california','colorado','connecticut',
    'delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa',
    'kansas','kentucky','louisiana','maine','maryland','massachusetts','michigan',
    'minnesota','mississippi','missouri','montana','nebraska','nevada',
    'new hampshire','new jersey','new mexico','new york','north carolina',
    'north dakota','ohio','oklahoma','oregon','pennsylvania','rhode island',
    'south carolina','south dakota','tennessee','texas','utah','vermont',
    'virginia','washington','west virginia','wisconsin','wyoming',
    'al','ak','az','ar','ca','co','ct','de','fl','ga','hi','id','il','in','ia',
    'ks','ky','la','me','md','ma','mi','mn','ms','mo','mt','ne','nv','nh','nj',
    'nm','ny','nc','nd','oh','ok','or','pa','ri','sc','sd','tn','tx','ut','vt',
    'va','wa','wv','wi','wy','dc',
}
MONTHS = {
    'january','february','march','april','may','june','july','august',
    'september','october','november','december',
    'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec',
}

In [17]:
def normalize_title(t):
    if not t or not isinstance(t, str): return ''
    t = t.lower().strip()
    t = re.sub(r'[-/\\|_]+', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    return re.sub(r'\s+', ' ', t).strip()

# normalize descriptions
def normalize_desc(t):
    if not t or not isinstance(t, str):
        return ''
    t = t.lower()
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'\b(19|20)\d{2}\b', '', t)
    t = re.sub(r'\bfy\s?\d{2,4}\b', '', t)
    t = re.sub(r'\bq[1-4]\b', '', t)
    t = re.sub(r'\bv\d+(\.\d+)*\b', '', t)
    t = re.sub(r'\b(version|edition|release|update|rev)\s*\d*\b', '', t)
    tokens = [tok for tok in t.split() if tok not in US_STATES and tok not in MONTHS]
    t = re.sub(r'[^a-z0-9\s]', '', ' '.join(tokens))
    return re.sub(r'\s+', ' ', t).strip()

In [18]:
# deduplicate per group
def dedup_group(group, threshold=0.7):
    if len(group) == 1:
        return group

    # pass 1: exact desc_norm match
    group = (
        group
        .sort_values('desc_len')
        .drop_duplicates('desc_norm', keep='first')
        .reset_index(drop=True)  # clean 0-based index for iloc below
    )
    if len(group) == 1:
        return group

    # pass 2: TF-IDF cosine on normalized descriptions
    descs = group['desc_norm'].tolist()
    if all(d.strip() == '' for d in descs):
        return group.iloc[[0]]

    tfidf = TfidfVectorizer(min_df=2, stop_words='english')
    try:
        X = tfidf.fit_transform(descs)
    except ValueError:
        return group.iloc[[0]]

    sims = cosine_similarity(X)

    kept = []
    absorbed = set()
    for i in range(len(group)):
        if i in absorbed:
            continue
        kept.append(i)
        for j in range(i + 1, len(group)):
            if sims[i, j] >= threshold:
                absorbed.add(j)

    cluster_map = {i: [i] for i in kept}
    for j in absorbed:
        for i in kept:
            if sims[i, j] >= threshold:
                cluster_map[i].append(j)
                break

    result_idxs = []
    for i, members in cluster_map.items():
        sub = group.iloc[members]
        med = sub['desc_len'].median()
        best = (sub['desc_len'] - med).abs().idxmin()
        result_idxs.append(best)

    return group.loc[result_idxs]


In [ ]:
# normalize titles
def topicalize_title(t):
    if not t or not isinstance(t, str):
        return ''
    t = t.lower().strip()
    t = re.sub(r'\b(19|20)\d{2}\b', '', t)
    t = re.sub(r'\bfy\s?\d{2,4}\b', '', t)
    t = re.sub(r'\bq[1-4]\b', '', t)
    t = re.sub(r'\bv\d+(\.\d+)*\b', '', t)
    t = re.sub(r'\b(version|edition|release|update|rev)\s*\d*\b', '', t)
    tokens = [tok for tok in t.split() if tok not in US_STATES and tok not in MONTHS]
    t = re.sub(r'[^a-z0-9\s]', '', ' '.join(tokens))
    return re.sub(r'\s+', ' ', t).strip()

# dedup per topic
def topical_dedup(df, threshold=0.7):
    df = df.copy()
    df['desc_len']  = df['description'].fillna('').str.len()
    df['desc_norm'] = df['description'].fillna('').apply(normalize_desc)

    has_topic = df['title_topic'].str.strip() != ''
    with_topic = df[has_topic]
    without_topic = df[~has_topic]

    deduped = (
        with_topic
        .groupby('title_topic', group_keys=False)
        .apply(lambda g: dedup_group(g, threshold=threshold))
        .reset_index(drop=True)
    )

    result = pd.concat([deduped, without_topic], ignore_index=True)
    print(f'{len(df):,} -> {len(result):,}  (removed {len(df)-len(result):,})')
    return result


Adding cols to our dfs

In [74]:
dgm['title_norm'] = dgm['dgm_title'].apply(normalize_title)
dgf['title_norm'] = dgf['title'].apply(normalize_title)
drp['title_norm'] = drp['project_title'].apply(normalize_title)

In [21]:
dgm['title_topic'] = dgm['title_norm'].apply(topicalize_title)
dgf['title_topic'] = dgf['title_norm'].apply(topicalize_title)
drp['title_topic'] = drp['title_norm'].apply(topicalize_title)

Differences look like this:
title_norm contains the normalized title

*   title_norm contains the normalized title
*   title_topic is normalized title with years also stripped

In [22]:
dgm[['dgm_title', 'title_norm', 'title_topic']]

,dgm_title,title_norm,title_topic
0,AFSC/RACE/SAP/Armistead: 1975 - 2016 eastern B...,afsc race sap armistead 1975 2016 eastern beri...,afsc race sap armistead eastern bering sea cra...
1,"TIGER/Line Shapefile, 2023, County, Moca Munic...",tiger line shapefile 2023 county moca municipi...,tiger line shapefile county moca municipio pr ...
2,FBSAB RECRUIT Reef Fish Belt Transect Survey a...,fbsab recruit reef fish belt transect survey a...,fbsab recruit reef fish belt transect survey a...
3,"Yellowknife, N. W. T., NT (CYZF)",yellowknife n w t nt cyzf,yellowknife n w t nt cyzf
4,L02194: NOS Hydrographic Survey,l02194 nos hydrographic survey,l02194 nos hydrographic survey
...,...,...,...
311815,OW NOAA AVHRR-GAC Sea-Surface Temperature,ow noaa avhrr gac sea surface temperature,ow noaa avhrr gac sea surface temperature
311816,"Mora County Blocks, Age by 5-Year Age Groups f...",mora county blocks age by 5 year age groups fo...,mora county blocks age by 5 year age groups fo...
311817,"TIGER/Line Shapefile, 2022, County, Brunswick ...",tiger line shapefile 2022 county brunswick cou...,tiger line shapefile county brunswick county t...
311818,SBUV2/NOAA-16 Level 2 Daily Ozone Profile and ...,sbuv2 noaa 16 level 2 daily ozone profile and ...,sbuv2 noaa 16 level 2 daily ozone profile and ...


In [23]:
drp['description'] = drp['dc_abstract']

In [25]:
def _token_index(titles):
    idx = defaultdict(set)
    for i, t in enumerate(titles):
        for tok in t.split():
            if len(tok) > 2:
                idx[tok].add(i)
    return idx

def match_drp_to_source(drp, src_df, title_col, src_name, score_cutoff=80):
    src_titles = src_df[title_col].fillna('').tolist()
    drp_titles = drp[title_col].fillna('').tolist()

    src_set = set(src_titles)
    src_index = _token_index(src_titles)

    exact, fuzzy = set(), {}
    for i, t in enumerate(drp_titles):
        if not t:
            continue
        if t in src_set:
            exact.add(i)
            continue
        # only compare against titles sharing more than 1 token
        candidate_idxs = set()
        for tok in t.split():
            if len(tok) > 2:
                candidate_idxs |= src_index.get(tok, set())
        if not candidate_idxs:
            continue
        candidates = [src_titles[j] for j in candidate_idxs]
        result = process.extractOne(t, candidates, scorer=fuzz.token_sort_ratio, score_cutoff=score_cutoff)
        if result:
            candidate_list = list(candidate_idxs)
            fuzzy[i] = (candidate_list[result[2]], result[1])

        if (i + 1) % 500 == 0:
            print(f"  {src_name}/{title_col}: {i+1:,}/{len(drp_titles):,}")

    print(f"  {src_name}/{title_col} — exact: {len(exact):,} | fuzzy: {len(fuzzy):,}")
    return exact, fuzzy


In [27]:
drp_dgm_exact, drp_dgm_fuzzy = match_drp_to_source(drp, dgm, 'title_norm',  'dgm')


  dgm/title_norm: 1,000/2,539
  dgm/title_norm: 1,500/2,539
  dgm/title_norm: 2,000/2,539
  dgm/title_norm — exact: 862 | fuzzy: 579


In [28]:
drp_dgm_topic_exact, drp_dgm_topic_fuzzy = match_drp_to_source(drp, dgm, 'title_topic', 'dgm')



  dgm/title_topic: 1,000/2,539
  dgm/title_topic: 1,500/2,539
  dgm/title_topic — exact: 940 | fuzzy: 550


In [29]:
drp_dgf_exact, drp_dgf_fuzzy = match_drp_to_source(drp, dgf, 'title_norm',  'dgf')


  dgf/title_norm: 1,000/2,539
  dgf/title_norm: 1,500/2,539
  dgf/title_norm — exact: 917 | fuzzy: 499


In [30]:
drp_dgf_topic_exact, drp_dgf_topic_fuzzy = match_drp_to_source(drp, dgf, 'title_topic', 'dgf')


  dgf/title_topic: 1,000/2,539
  dgf/title_topic: 1,500/2,539
  dgf/title_topic — exact: 955 | fuzzy: 512


Putting everything together

In [31]:
def build_match_df(drp, norm_exact, norm_fuzzy, norm_src_df, norm_title_col,
                    topic_exact, topic_fuzzy, topic_src_df, topic_title_col,
                    source_name, desc_col):
    all_idx = set(norm_exact) | set(norm_fuzzy) | set(topic_exact) | set(topic_fuzzy)
    rows = []

    for i in sorted(all_idx):
        drp_row = drp.iloc[i]
        row = {
            'project_title':  drp_row['project_title'],
            'drp_title_norm': drp_row['title_norm'],
            'drp_title_topic': drp_row['title_topic'],
            'dc_abstract': drp_row['dc_abstract'],
        }

        # norm match
        norm_info = {
            f'{source_name}_title': None,
            f'{source_name}_title_norm': None,
            f'{source_name}_title_topic': None,
            f'{source_name}_norm_match_type': None,
            f'{source_name}_norm_match_score': None,
            f'{source_name}_description': None,
        }
        if i in norm_exact:
            matching_rows = norm_src_df[norm_src_df['title_norm'] == drp_row['title_norm']]
            if not matching_rows.empty:
                mr = matching_rows.iloc[0]
                norm_info.update({
                    f'{source_name}_title': mr[norm_title_col],
                    f'{source_name}_title_norm': mr['title_norm'],
                    f'{source_name}_title_topic': mr['title_topic'],
                    f'{source_name}_norm_match_type': 'exact',
                    f'{source_name}_norm_match_score': 100,
                    f'{source_name}_description': mr[desc_col],
                })
        elif i in norm_fuzzy:
            src_idx, score = norm_fuzzy[i]
            mr = norm_src_df.iloc[src_idx]
            norm_info.update({
                f'{source_name}_title': mr[norm_title_col],
                f'{source_name}_title_norm': mr['title_norm'],
                f'{source_name}_title_topic': mr['title_topic'],
                f'{source_name}_norm_match_type': 'fuzzy',
                f'{source_name}_norm_match_score': score,
                f'{source_name}_description': mr[desc_col],
            })
        row.update(norm_info)

        # topic match
        topic_info = {
            f'{source_name}_topic_title':  None,
            f'{source_name}_topic_title_norm': None,
            f'{source_name}_topic_title_topic': None,
            f'{source_name}_topic_match_type': None,
            f'{source_name}_topic_match_score': None,
        }
        if i in topic_exact:
            matching_rows = topic_src_df[topic_src_df['title_topic'] == drp_row['title_topic']]
            if not matching_rows.empty:
                mr = matching_rows.iloc[0]
                topic_info.update({
                    f'{source_name}_topic_title': mr[topic_title_col],
                    f'{source_name}_topic_title_norm': mr['title_norm'],
                    f'{source_name}_topic_title_topic': mr['title_topic'],
                    f'{source_name}_topic_match_type': 'exact',
                    f'{source_name}_topic_match_score': 100,
                })
                if row[f'{source_name}_description'] is None:
                    row[f'{source_name}_description'] = mr[desc_col]
        elif i in topic_fuzzy:
            src_idx, score = topic_fuzzy[i]
            mr = topic_src_df.iloc[src_idx]
            topic_info.update({
                f'{source_name}_topic_title': mr[topic_title_col],
                f'{source_name}_topic_title_norm':  mr['title_norm'],
                f'{source_name}_topic_title_topic': mr['title_topic'],
                f'{source_name}_topic_match_type': 'fuzzy',
                f'{source_name}_topic_match_score': score,
            })
            if row[f'{source_name}_description'] is None:
                row[f'{source_name}_description'] = mr[desc_col]
        row.update(topic_info)

        rows.append(row)
    return pd.DataFrame(rows)


dgm_matches = build_match_df(
    drp,
    drp_dgm_exact, drp_dgm_fuzzy, dgm, 'dgm_title',
    drp_dgm_topic_exact, drp_dgm_topic_fuzzy, dgm, 'dgm_title',
    'dgm', 'dgm_notes',
)

dgf_matches = build_match_df(
    drp,
    drp_dgf_exact, drp_dgf_fuzzy, dgf, 'title',
    drp_dgf_topic_exact, drp_dgf_topic_fuzzy, dgf, 'title',
    'dgf', 'description',
)

# merge into one df for review — outer so we keep entries matched by only one source
merge_keys = ['project_title', 'drp_title_norm', 'drp_title_topic', 'dc_abstract']
matches = dgm_matches.merge(dgf_matches, on=merge_keys, how='outer')

print(f"Total DRP entries with any match: {len(matches):,}")
for src in ['dgm', 'dgf']:
    has_match = matches[f'{src}_norm_match_type'].notna() | matches[f'{src}_topic_match_type'].notna()
    print(f"  {src.upper()}: {has_match.sum():,} matched")
    print(f"    norm  — exact: {(matches[f'{src}_norm_match_type']=='exact').sum():,}  fuzzy: {(matches[f'{src}_norm_match_type']=='fuzzy').sum():,}")
    print(f"    topic — exact: {(matches[f'{src}_topic_match_type']=='exact').sum():,}  fuzzy: {(matches[f'{src}_topic_match_type']=='fuzzy').sum():,}")


Total DRP entries with any match: 1,545
  DGM: 1,506 matched
    norm  — exact: 872  fuzzy: 579
    topic — exact: 950  fuzzy: 550
  DGF: 1,480 matched
    norm  — exact: 927  fuzzy: 499
    topic — exact: 965  fuzzy: 512


### Embeddings for similarity calc

In [36]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
nomic_model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True, device=device)

<All keys matched successfully>


In [37]:
tok = nomic_model.tokenizer
sample = (matches['dgf_description'].fillna('').astype(str)
          .sample(min(5000, len(matches)), random_state=0).tolist())
lens = np.array([len(tok.encode(s)) for s in sample])
print(f'desc tokens — median {int(np.median(lens))}  p95 {int(np.percentile(lens,95))}  '
      f'p99 {int(np.percentile(lens,99))}  max {int(lens.max())}  >2048 {(lens>2048).mean():.1%}')


desc tokens — median 107  p95 669  p99 783  max 1735  >2048 0.0%


In [38]:
nomic_model.max_seq_length = 2048


In [39]:
PREFIX = 'classification: '
def embed(col, df):
    texts = [PREFIX + t for t in df[col].fillna('').astype(str).tolist()]
    return nomic_model.encode(texts, batch_size=8, show_progress_bar=True,
                              normalize_embeddings=True, convert_to_numpy=True)

Extracting embeddings

In [40]:
drp_vecs = embed('dc_abstract', matches)

Batches: 100%|██████████| 194/194 [04:03<00:00,  1.26s/it]


In [41]:
dgm_vecs = embed('dgm_description', matches)


Batches: 100%|██████████| 194/194 [03:09<00:00,  1.02it/s]


In [42]:
dgf_vecs = embed('dgf_description', matches)


Batches:   0%|          | 0/194 [00:00<?, ?it/s]

Batches: 100%|██████████| 194/194 [03:02<00:00,  1.06it/s]


In [43]:
# normalize_embeddings=True so cosine sim = dot product
matches['sim_drp_dgm'] = (drp_vecs * dgm_vecs).sum(axis=1)
matches['sim_drp_dgf'] = (drp_vecs * dgf_vecs).sum(axis=1)
matches['sim_dgf_dgm'] = (dgm_vecs * dgf_vecs).sum(axis=1)

# NaN where there was no match (embedding was on empty string)
matches.loc[matches['dgm_description'].isna(), 'sim_drp_dgm'] = np.nan
matches.loc[matches['dgf_description'].isna(), 'sim_drp_dgf'] = np.nan

# also for comparison of dgf vs dgm
matches.loc[matches['dgm_description'].isna(), 'sim_dgf_dgm'] = np.nan
matches.loc[matches['dgf_description'].isna(), 'sim_dgf_dgm'] = np.nan

In [44]:
drp_notes_lookup = drp.drop_duplicates('project_title').set_index('project_title')['notes']
matches['drp_notes'] = matches['project_title'].map(drp_notes_lookup)

In [ ]:
matches.to_parquet(str(DATA_DIR / '02_01_all_matches.parquet'), index=False)
print(f"Saved {len(matches):,} rows to 02_01_all_matches.parquet")

Saved 1,545 rows to 03_01_all_matches.parquet


In [ ]:
matches.to_excel(str(DATA_DIR / '02_01_all_matches.xlsx'))

/var/folders/f0/g61h031s7yscf43_m6xj3l0c0000gn/T/ipykernel_29158/1556573742.py:1: UserWarning: Cell contents too long (34242), truncated to 32767 characters
  matches.to_excel(str(DATA_DIR / '03_01_all_matches.xlsx'))
/var/folders/f0/g61h031s7yscf43_m6xj3l0c0000gn/T/ipykernel_29158/1556573742.py:1: UserWarning: Cell contents too long (61399), truncated to 32767 characters
  matches.to_excel(str(DATA_DIR / '03_01_all_matches.xlsx'))


## Difference analysis

### N-grams and tf-idf

In [47]:
matches.columns

Index(['project_title', 'drp_title_norm', 'drp_title_topic', 'dc_abstract',
       'dgm_title', 'dgm_title_norm', 'dgm_title_topic', 'dgm_norm_match_type',
       'dgm_norm_match_score', 'dgm_description', 'dgm_topic_title',
       'dgm_topic_title_norm', 'dgm_topic_title_topic', 'dgm_topic_match_type',
       'dgm_topic_match_score', 'dgf_title', 'dgf_title_norm',
       'dgf_title_topic', 'dgf_norm_match_type', 'dgf_norm_match_score',
       'dgf_description', 'dgf_topic_title', 'dgf_topic_title_norm',
       'dgf_topic_title_topic', 'dgf_topic_match_type',
       'dgf_topic_match_score', 'sim_drp_dgm', 'sim_drp_dgf', 'sim_dgf_dgm',
       'drp_notes'],
      dtype='str')

Finfing unique ngrams in drp that dont appear in either source

In [48]:
def get_ngrams(text, n=3):
    if not isinstance(text, str) or not text.strip():
        return set()
    words = re.findall(r'\w+', text.lower())
    return set(' '.join(words[i:i+n]) for i in range(len(words)-n+1))

unique_ngrams_vs_dgm = Counter()
unique_ngrams_vs_dgf = Counter()

for _, row in matches.iterrows():
    drp_ngrams = get_ngrams(row['dc_abstract'])
    dgm_ngrams = get_ngrams(row['dgm_description'])
    dgf_ngrams = get_ngrams(row['dgf_description'])

    if dgm_ngrams:
        unique_ngrams_vs_dgm.update(drp_ngrams - dgm_ngrams)
    if dgf_ngrams:
        unique_ngrams_vs_dgf.update(drp_ngrams - dgf_ngrams)

In [49]:
print('Top 3-grams in drp abstracts NOT in dgm:')
for ng, count in unique_ngrams_vs_dgm.most_common(30):
    print(f'  [{count:,}] {ng}')

Top 3-grams in drp abstracts NOT in dgm:
  [68] electronic navigational charts
  [67] are produced for
  [67] located in the
  [66] in agreement with
  [66] navigable waterways which
  [66] the central united
  [66] e g maintained
  [66] by usace for
  [66] navigation by usace
  [66] iencs coverage area
  [66] agreement with noaa
  [66] does not produce
  [66] inland waterways that
  [66] vessels e g
  [66] waterways that are
  [66] which the national
  [66] at a depth
  [66] of 9 14
  [66] in the central
  [66] commercially navigable waterways
  [66] to inland waterways
  [66] noaa does not
  [66] encs however special
  [66] for navigation by
  [66] the waterway project
  [66] miles across 21
  [66] 21 rivers primarily
  [66] may be produced
  [66] iencs may be
  [66] iencs apply to


In [50]:
print('Top 3-grams in drp abstracts NOT in dgf:')
for ng, count in unique_ngrams_vs_dgf.most_common(30):
    print(f'  [{count:,}] {ng}')

Top 3-grams in drp abstracts NOT in dgf:
  [68] electronic navigational charts
  [67] are produced for
  [67] located in the
  [66] in agreement with
  [66] navigable waterways which
  [66] the central united
  [66] e g maintained
  [66] by usace for
  [66] navigation by usace
  [66] iencs coverage area
  [66] agreement with noaa
  [66] does not produce
  [66] inland waterways that
  [66] vessels e g
  [66] waterways that are
  [66] which the national
  [66] at a depth
  [66] of 9 14
  [66] in the central
  [66] commercially navigable waterways
  [66] to inland waterways
  [66] noaa does not
  [66] encs however special
  [66] for navigation by
  [66] the waterway project
  [66] miles across 21
  [66] 21 rivers primarily
  [66] may be produced
  [66] iencs may be
  [66] iencs apply to


how much new content does drp add?

In [51]:
def new_content_ratio(drp_text, other_text):
    if not isinstance(drp_text, str) or not drp_text.strip() or not isinstance(other_text, str) or not other_text.strip():
        return None
    drp_words = set(re.findall(r'\w+', drp_text.lower()))
    other_words = set(re.findall(r'\w+', other_text.lower()))
    if not drp_words:
        return None
    return len(drp_words - other_words) / len(drp_words)

matches['new_content_vs_dgm'] = matches.apply(lambda r: new_content_ratio(r['dc_abstract'], r['dgm_description']), axis=1)
matches['new_content_vs_dgf'] = matches.apply(lambda r: new_content_ratio(r['dc_abstract'], r['dgf_description']), axis=1)

print('New content ratios')
print('vs dgm:')
print(matches['new_content_vs_dgm'].describe().to_string())
print('\nvs dgf:')
print(matches['new_content_vs_dgf'].describe().to_string())

New content ratios
vs dgm:
count    1506.000000
mean        0.336042
std         0.373692
min         0.000000
25%         0.014519
50%         0.121071
75%         0.750000
max         1.000000

vs dgf:
count    1480.000000
mean        0.330561
std         0.372872
min         0.000000
25%         0.014085
50%         0.105101
75%         0.727273
max         1.000000


rows with high new content

In [52]:
leakage_risk = matches[(matches['new_content_vs_dgm'] > 0.3) | (matches['new_content_vs_dgf'] > 0.3)].copy()
print(f'>30% new content: {len(leakage_risk):,}')

>30% new content: 624


new sentences

In [53]:
def new_sentences(drp_text, other_text):
    if not isinstance(drp_text, str) or not drp_text.strip() or not isinstance(other_text, str) or not other_text.strip():
        return []
    drp_sents = [s.strip() for s in re.split(r'[.!?]+', drp_text) if s.strip()]
    other_words = set(re.findall(r'\w+', other_text.lower()))
    new = []
    for s in drp_sents:
        s_words = set(re.findall(r'\w+', s.lower()))
        if not s_words:
            continue
        overlap = len(s_words & other_words) / len(s_words)
        if overlap < 0.5:
            new.append(s)
    return new

examples

In [ ]:
for _, row in leakage_risk.head(20).iterrows():
    new_vs_dgm = new_sentences(row['dc_abstract'], row['dgm_description'])
    new_vs_dgf = new_sentences(row['dc_abstract'], row['dgf_description'])
    if new_vs_dgm or new_vs_dgf:
        print(f'\n  Title: {row['project_title'][:80]}')
        for s in new_vs_dgm[:3]:
            print(f'    [new vs dgm] {s[:120]}')
        for s in new_vs_dgf[:3]:
            print(f'    [new vs dgf] {s[:120]}')

matches.to_parquet('02_02_drp_leakage_analysis.parquet', index=False)
print(f'\nSaved {len(matches):,} rows to 03_02_drp_leakage_analysis.parquet')


  Title: 2006 IUR Public Database
    [new vs dgm] This submission includes publicly available data extracted in its original form
    [new vs dgm] Please reference the Related Publication listed here for source and citation information &lt;br /&gt;&lt;br /&gt;

The f
    [new vs dgm] Please note that no information claimed as TSCA Confidential Business Information by an IUR reporter is contained in thi
    [new vs dgf] This submission includes publicly available data extracted in its original form
    [new vs dgf] Please reference the Related Publication listed here for source and citation information &lt;br /&gt;&lt;br /&gt;

The f
    [new vs dgf] Please note that no information claimed as TSCA Confidential Business Information by an IUR reporter is contained in thi

  Title: 2014 Minority Veteran Report
    [new vs dgm] This project includes a pdf capture of a webpage and the underlying data for the visualizations
    [new vs dgm] <br><br>It is about the 2014 Minority Veteran Repo

### How similar are data.gov and data.gov mirror descriptions?

In [55]:
matches['sim_dgf_dgm'].describe()

count    1441.000000
mean        0.991153
std         0.037783
min         0.456155
25%         0.999912
50%         1.000000
75%         1.000000
max         1.000000
Name: sim_dgf_dgm, dtype: float64

## Exploring text characteristics on full dataset

In [56]:
dgf.columns

Index(['title', 'description', 'publisher', 'identifier', 'slug',
       'has_spatial', 'popularity', 'last_harvested_date', 'keyword', 'theme',
       'org_name', 'org_type', 'org_id', 'org_slug', 'org_description',
       'dcat_issued', 'dcat_modified', 'dcat_access_level',
       'dcat_access_comment', 'dcat_landing_page', 'dcat_language',
       'dcat_temporal', 'dcat_spatial', 'dcat_bureau_code',
       'dcat_program_code', 'dcat_periodicity', 'dcat_license', 'dcat_rights',
       'title_norm', 'title_topic'],
      dtype='str')

In [57]:
# Load full datasets
dgf = pd.read_parquet(str(DATA_DIR / '02_dgf.parquet'))

drp_text = drp['dc_abstract'].fillna('').astype(str)
dgf_text = dgf['description'].fillna('').astype(str)

print(f"DRP: {len(drp):,} records")
print(f"DGF: {len(dgf):,} records")


DRP: 2,539 records
DGF: 514,352 records


basic stats

In [58]:
for name, col in [('DRP', drp_text), ('DGF', dgf_text)]:
    lengths = col.str.len()
    words = col.str.split().str.len()
    empty = (col == '').sum()
    print(f"\n{name}:")
    print(f"  Empty: {empty:,} ({empty/len(col)*100:.1f}%)")
    print(f"  Char length: mean={lengths.mean():.0f}, median={lengths.median():.0f}, max={lengths.max():.0f}")
    print(f"  Word count:  mean={words.mean():.0f}, median={words.median():.0f}, max={words.max():.0f}")


DRP:
  Empty: 0 (0.0%)
  Char length: mean=1373, median=777, max=61399
  Word count:  mean=196, median=108, max=9262

DGF:
  Empty: 0 (0.0%)
  Char length: mean=1142, median=1100, max=30886
  Word count:  mean=169, median=161, max=3235


html and formatting

In [59]:
artifacts = {
    'HTML <br> tags': r'<br',
    'HTML <b> tags': r'<b>',
    'HTML &amp;': r'&amp;',
    'Any HTML tag': r'<[a-zA-Z]',
    'URLs': r'https?://',
    'File references (.csv/.pdf/.zip etc)': r'\.\b(csv|pdf|zip|xlsx|json|xml)\b',
}
for label, pattern in artifacts.items():
    drp_count = drp_text.str.contains(pattern, regex=True, na=False).sum()
    dgf_count = dgf_text.str.contains(pattern, regex=True, na=False).sum()
    print(f"  {label}:")
    print(f"    DRP: {drp_count:,} ({drp_count/len(drp)*100:.1f}%)")
    print(f"    DGF: {dgf_count:,} ({dgf_count/len(dgf)*100:.1f}%)")

  HTML <br> tags:
    DRP: 766 (30.2%)
    DGF: 4,030 (0.8%)
  HTML <b> tags:
    DRP: 124 (4.9%)
    DGF: 1,152 (0.2%)
  HTML &amp;:
    DRP: 171 (6.7%)
    DGF: 1,738 (0.3%)
  Any HTML tag:
    DRP: 767 (30.2%)
    DGF: 12,811 (2.5%)
  URLs:
    DRP: 855 (33.7%)
    DGF: 29,295 (5.7%)
  File references (.csv/.pdf/.zip etc):
    DRP: 318 (12.5%)
    DGF: 7,908 (1.5%)


/var/folders/f0/g61h031s7yscf43_m6xj3l0c0000gn/T/ipykernel_29158/2665097938.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  drp_count = drp_text.str.contains(pattern, regex=True, na=False).sum()
/var/folders/f0/g61h031s7yscf43_m6xj3l0c0000gn/T/ipykernel_29158/2665097938.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  dgf_count = dgf_text.str.contains(pattern, regex=True, na=False).sum()


drp specific boilerplate

In [60]:
boilerplate = [
    'this project includes',
    'pdf capture of a webpage',
    'underlying data for the visualizations',
    'datalumos',
    'updated version from this record',
    'data rescue',
    'data at risk',
    'end of term',
]
for phrase in boilerplate:
    drp_count = drp_text.str.lower().str.contains(phrase, na=False).sum()
    dgf_count = dgf_text.str.lower().str.contains(phrase, na=False).sum()
    print(f"  '{phrase}':")
    print(f"    DRP: {drp_count:,} ({drp_count/len(drp)*100:.1f}%)  |  DGF: {dgf_count:,} ({dgf_count/len(dgf)*100:.1f}%)")

  'this project includes':
    DRP: 15 (0.6%)  |  DGF: 10 (0.0%)
  'pdf capture of a webpage':
    DRP: 13 (0.5%)  |  DGF: 0 (0.0%)
  'underlying data for the visualizations':
    DRP: 14 (0.6%)  |  DGF: 0 (0.0%)
  'datalumos':
    DRP: 13 (0.5%)  |  DGF: 0 (0.0%)
  'updated version from this record':
    DRP: 1 (0.0%)  |  DGF: 0 (0.0%)
  'data rescue':
    DRP: 0 (0.0%)  |  DGF: 6 (0.0%)
  'data at risk':
    DRP: 0 (0.0%)  |  DGF: 4 (0.0%)
  'end of term':
    DRP: 0 (0.0%)  |  DGF: 0 (0.0%)


distinctive vocabulary

In [61]:
def word_freq(texts, min_count=10):
    counter = Counter()
    for t in texts:
        counter.update(set(re.findall(r'\b[a-z]{3,}\b', t.lower())))
    return counter

drp_freq = word_freq(drp_text)
dgf_freq = word_freq(dgf_text)

# Normalize by dataset size
drp_n = len(drp)
dgf_n = len(dgf)

words much more common in drp than dgf

In [62]:
scores = {}
for word, count in drp_freq.items():
    drp_rate = count / drp_n
    dgf_rate = dgf_freq.get(word, 0) / dgf_n
    if dgf_rate > 0 and count >= 20:
        scores[word] = drp_rate / dgf_rate
    elif dgf_rate == 0 and count >= 20:
        scores[word] = float('inf')

for word, score in sorted(scores.items(), key=lambda x: -x[1])[:30]:
    drp_pct = drp_freq[word] / drp_n * 100
    dgf_pct = dgf_freq.get(word, 0) / dgf_n * 100
    print(f"  {word:30s} DRP: {drp_pct:5.1f}%  DGF: {dgf_pct:5.2f}%  ratio: {score:.1f}x")

  climatecafe                    DRP:   1.1%  DGF:  0.00%  ratio: infx
  pudl                           DRP:   0.8%  DGF:  0.00%  ratio: infx
  frictionless                   DRP:   0.8%  DGF:  0.00%  ratio: infx
  projectionsdata                DRP:   0.9%  DGF:  0.00%  ratio: infx
  programdata                    DRP:   0.9%  DGF:  0.00%  ratio: infx
  grantsdata                     DRP:   0.9%  DGF:  0.00%  ratio: infx
  systemdata                     DRP:   0.9%  DGF:  0.00%  ratio: infx
  geographyhrsa                  DRP:   0.9%  DGF:  0.00%  ratio: infx
  sitesdata                      DRP:   0.9%  DGF:  0.00%  ratio: infx
  dashboardsdata                 DRP:   0.9%  DGF:  0.00%  ratio: infx
  practicesnational              DRP:   0.9%  DGF:  0.00%  ratio: infx
  bureaudata                     DRP:   0.9%  DGF:  0.00%  ratio: infx
  clincian                       DRP:   0.9%  DGF:  0.00%  ratio: infx
  programsdata                   DRP:   0.9%  DGF:  0.00%  ratio: infx
  bhw 

words much more common in dgf than drp

In [64]:
scores_rev = {}
for word, count in dgf_freq.items():
    dgf_rate = count / dgf_n
    drp_rate = drp_freq.get(word, 0) / drp_n
    if drp_rate > 0 and count >= 100:
        scores_rev[word] = dgf_rate / drp_rate
    elif drp_rate == 0 and count >= 100:
        scores_rev[word] = float('inf')

for word, score in sorted(scores_rev.items(), key=lambda x: -x[1])[:30]:
    dgf_pct = dgf_freq[word] / dgf_n * 100
    drp_pct = drp_freq.get(word, 0) / drp_n * 100
    print(f"  {word:30s} DGF: {dgf_pct:5.1f}%  DRP: {drp_pct:5.2f}%  ratio: {score:.1f}x")

  desk                           DGF:   0.0%  DRP:  0.00%  ratio: infx
  holder                         DGF:   0.0%  DRP:  0.00%  ratio: infx
  emitted                        DGF:   0.1%  DRP:  0.00%  ratio: infx
  dating                         DGF:   0.1%  DRP:  0.00%  ratio: infx
  arrests                        DGF:   0.1%  DRP:  0.00%  ratio: infx
  jpl                            DGF:   0.1%  DRP:  0.00%  ratio: infx
  asteroids                      DGF:   0.1%  DRP:  0.00%  ratio: infx
  browser                        DGF:   0.0%  DRP:  0.00%  ratio: infx
  placement                      DGF:   0.1%  DRP:  0.00%  ratio: infx
  fresh                          DGF:   0.1%  DRP:  0.00%  ratio: infx
  intersection                   DGF:   0.1%  DRP:  0.00%  ratio: infx
  citywide                       DGF:   0.0%  DRP:  0.00%  ratio: infx
  explored                       DGF:   0.1%  DRP:  0.00%  ratio: infx
  realtime                       DGF:   0.0%  DRP:  0.00%  ratio: infx
  succ

agency distribution

In [63]:
agency_keywords = [
    'noaa', 'nasa', 'epa', 'usgs', 'fema', 'cdc', 'hud', 'usda',
    'dot', 'doe', 'census', 'fda', 'cms', 'nih', 'va ', 'veteran',
    'defense', 'education', 'labor', 'commerce', 'interior',
]
for kw in agency_keywords:
    drp_count = drp_text.str.lower().str.contains(kw, na=False).sum()
    dgf_count = dgf_text.str.lower().str.contains(kw, na=False).sum()
    if drp_count > 10 or dgf_count > 100:
        drp_pct = drp_count / drp_n * 100
        dgf_pct = dgf_count / dgf_n * 100
        ratio = drp_pct / dgf_pct if dgf_pct > 0 else float('inf')
        print(f"  {kw:15s} DRP: {drp_pct:5.1f}%  DGF: {dgf_pct:5.2f}%  ratio: {ratio:.1f}x")

  noaa            DRP:   5.4%  DGF:  9.62%  ratio: 0.6x
  nasa            DRP:   0.0%  DGF:  1.48%  ratio: 0.0x
  epa             DRP:  29.3%  DGF: 22.39%  ratio: 1.3x
  usgs            DRP:   3.2%  DGF:  5.66%  ratio: 0.6x
  fema            DRP:   3.7%  DGF:  0.27%  ratio: 13.8x
  cdc             DRP:  26.5%  DGF:  0.23%  ratio: 114.0x
  hud             DRP:   2.0%  DGF:  0.11%  ratio: 19.0x
  usda            DRP:   0.3%  DGF:  0.51%  ratio: 0.6x
  dot             DRP:   1.9%  DGF:  0.32%  ratio: 5.9x
  doe             DRP:   9.0%  DGF:  2.54%  ratio: 3.5x
  census          DRP:  11.3%  DGF: 55.86%  ratio: 0.2x
  fda             DRP:   0.5%  DGF:  0.03%  ratio: 15.8x
  cms             DRP:   1.7%  DGF:  0.26%  ratio: 6.7x
  nih             DRP:   0.1%  DGF:  0.03%  ratio: 2.3x
  va              DRP:   3.6%  DGF:  0.33%  ratio: 10.9x
  veteran         DRP:   7.7%  DGF:  0.23%  ratio: 32.9x
  defense         DRP:   0.4%  DGF:  0.07%  ratio: 5.4x
  education       DRP:   4.8%  DGF:  2.48

# Checked excel, creating DF

In [ ]:
df_match = pd.read_excel(str(DATA_DIR / '02_01_all_matches_checked.xlsx'))

In [66]:
df_match.columns

Index(['Column1', 'project_title', 'dgm_title', 'dgf_title',
       'dgm_norm_match_type', 'dgm_norm_match_score', 'dgf_norm_match_type',
       'dgf_norm_match_score', 'drp_title_topic', 'dgm_title_topic',
       'dgf_title_topic', 'dgm_topic_title', 'dgf_topic_title',
       'dgm_topic_title_norm', 'dgf_topic_title_norm', 'dgm_topic_title_topic',
       'dgf_topic_title_topic', 'dgm_topic_match_type',
       'dgm_topic_match_score', 'dgf_topic_match_type',
       'dgf_topic_match_score', 'drp_notes', 'dc_abstract', 'dgm_description',
       'dgf_description', 'sim_drp_dgm', 'sim_drp_dgf', 'sim_dgf_dgm',
       'checked'],
      dtype='str')

In [67]:
checked_titles = set(df_match.loc[df_match['checked'] != 0, 'project_title'])
matches_checked = matches[matches['project_title'].isin(checked_titles)].reset_index(drop=True)

print(f'df_match rows with check != 0: {len(checked_titles):,}')
print(f'matches before filter:          {len(matches):,}')
print(f'matches after filter:           {len(matches_checked):,}')
matches_checked

df_match rows with check != 0: 1,410
matches before filter:          1,545
matches after filter:           1,443


,project_title,drp_title_norm,drp_title_topic,dc_abstract,dgm_title,dgm_title_norm,dgm_title_topic,dgm_norm_match_type,dgm_norm_match_score,dgm_description,...,dgf_topic_title_norm,dgf_topic_title_topic,dgf_topic_match_type,dgf_topic_match_score,sim_drp_dgm,sim_drp_dgf,sim_dgf_dgm,drp_notes,new_content_vs_dgm,new_content_vs_dgf
0,1998-2023 Serotype Data for Invasive Pneumococ...,1998 2023 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,CDC monitors invasive bacterial infections tha...,1998-2021 Serotype Data for Invasive Pneumococ...,1998 2021 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,fuzzy,99.137931,CDC monitors invasive bacterial infections tha...,...,1998 2022 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,exact,100.0,0.989752,0.988295,0.999354,NaN,0.150442,0.150442
1,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,The map illustrates the total number of 2013 a...,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,exact,100.000000,The map illustrates the total number of 2013 a...,...,2013 2014 phap associates by state,phap associates by state,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000
2,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,This project includes a pdf capture of a webpa...,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,exact,100.000000,"Over the past 30 years, racial and ethnic mino...",...,minority veteran report,minority veteran report,exact,100.0,0.670759,0.670759,1.000000,NaN,0.875000,0.875000
3,2016 AmeriCorps MES AmeriCorps Member Exit Survey,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2016 AmeriCorps MES: AmeriCorps Member Exit Su...,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,2023 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.0,0.971931,0.971931,1.000000,NaN,0.011628,0.011628
4,2017 AmeriCorps MES AmeriCorps Member Exit Survey,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2017 AmeriCorps MES: AmeriCorps Member Exit Su...,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,2023 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.0,0.972655,0.972655,1.000000,NaN,0.011628,0.011628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1438,Women Veterans Forum,women veterans forum,women veterans forum,This project includes a pdf capture of a webpa...,Women Veterans Forum,women veterans forum,women veterans forum,exact,100.000000,"On February 7, 2017, the VA Office of Enterpri...",...,women veterans forum,women veterans forum,exact,100.0,0.851454,0.851454,1.000000,NaN,0.694444,0.694444
1439,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,The Youth Risk Behavior Surveillance System (Y...,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,exact,100.000000,The Youth Risk Behavior Surveillance System (Y...,...,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000
1440,Youth Risk Behavior Surveillance System (YRBSS...,youth risk behavior surveillance system yrbss ...,youth risk behavior surveillance system yrbss,YRBSS monitors adolescent health behavior chan...,

In [68]:
dupes = matches_checked[matches_checked.duplicated('project_title', keep=False)]
dupes = dupes.sort_values('project_title')
print(f'Duplicate project_title rows: {len(dupes):,}  ({dupes["project_title"].nunique():,} unique titles)')
dupes


Duplicate project_title rows: 52  (19 unique titles)


,project_title,drp_title_norm,drp_title_topic,dc_abstract,dgm_title,dgm_title_norm,dgm_title_topic,dgm_norm_match_type,dgm_norm_match_score,dgm_description,...,dgf_topic_title_norm,dgf_topic_title_topic,dgf_topic_match_type,dgf_topic_match_score,sim_drp_dgm,sim_drp_dgf,sim_dgf_dgm,drp_notes,new_content_vs_dgm,new_content_vs_dgf
84,AH Provisional COVID-19 Deaths by Week and Age,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,Deaths involving coronavirus disease 2019 (COV...,AH Provisional COVID-19 Deaths by Week and Age,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,exact,100.000000,Deaths involving coronavirus disease 2019 (COV...,...,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,exact,100.000000,1.000000,1.000000,1.000000,NaN,0.000000,0.000000
85,AH Provisional COVID-19 Deaths by Week and Age,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,Provisional counts of deaths involving coronav...,AH Provisional COVID-19 Deaths by Week and Age,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,exact,100.000000,Deaths involving coronavirus disease 2019 (COV...,...,ah provisional covid 19 deaths by week and age,ah provisional covid 19 deaths by week and age,exact,100.000000,0.913950,0.913950,1.000000,NaN,0.315789,0.315789
259,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,This dataset provides modeled predictions of P...,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,This dataset provides modeled predictions of P...,...,daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,0.999910,0.999910,1.000000,NaN,0.014085,0.014085
260,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,This dataset provides modeled predictions of P...,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,This dataset provides modeled predictions of P...,...,daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,0.999910,0.999910,1.000000,NaN,0.014085,0.014085
261,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,This dataset provides modeled predictions of P...,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,This dataset provides modeled predictions of P...,...,daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,0.999910,0.999910,1.000000,NaN,0.014085,0.014085
262,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,This dataset provides modeled predictions of P...,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,This dataset provides modeled predictions of P...,...,daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,exact,100.000000,0.999910,0.999910,1.000000,NaN,0.014085,0.014085
263,"Daily Census Tract-Level PM2.5 Concentrations,...",daily census tract level pm25 concentrations 2...,daily census tract level pm25 concentrations,This dataset provides modeled predictions of P...,"Daily Census Tra

In [69]:
matches_checked = matches_checked.drop_duplicates('project_title', keep='first').reset_index(drop=True)
print(f'{len(matches_checked):,} rows after dedup')
matches_checked


1,410 rows after dedup


,project_title,drp_title_norm,drp_title_topic,dc_abstract,dgm_title,dgm_title_norm,dgm_title_topic,dgm_norm_match_type,dgm_norm_match_score,dgm_description,...,dgf_topic_title_norm,dgf_topic_title_topic,dgf_topic_match_type,dgf_topic_match_score,sim_drp_dgm,sim_drp_dgf,sim_dgf_dgm,drp_notes,new_content_vs_dgm,new_content_vs_dgf
0,1998-2023 Serotype Data for Invasive Pneumococ...,1998 2023 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,CDC monitors invasive bacterial infections tha...,1998-2021 Serotype Data for Invasive Pneumococ...,1998 2021 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,fuzzy,99.137931,CDC monitors invasive bacterial infections tha...,...,1998 2022 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,exact,100.0,0.989752,0.988295,0.999354,NaN,0.150442,0.150442
1,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,The map illustrates the total number of 2013 a...,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,exact,100.000000,The map illustrates the total number of 2013 a...,...,2013 2014 phap associates by state,phap associates by state,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000
2,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,This project includes a pdf capture of a webpa...,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,exact,100.000000,"Over the past 30 years, racial and ethnic mino...",...,minority veteran report,minority veteran report,exact,100.0,0.670759,0.670759,1.000000,NaN,0.875000,0.875000
3,2016 AmeriCorps MES AmeriCorps Member Exit Survey,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2016 AmeriCorps MES: AmeriCorps Member Exit Su...,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,2023 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.0,0.971931,0.971931,1.000000,NaN,0.011628,0.011628
4,2017 AmeriCorps MES AmeriCorps Member Exit Survey,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2017 AmeriCorps MES: AmeriCorps Member Exit Su...,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,2023 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.0,0.972655,0.972655,1.000000,NaN,0.011628,0.011628
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,Women Veterans Forum,women veterans forum,women veterans forum,This project includes a pdf capture of a webpa...,Women Veterans Forum,women veterans forum,women veterans forum,exact,100.000000,"On February 7, 2017, the VA Office of Enterpri...",...,women veterans forum,women veterans forum,exact,100.0,0.851454,0.851454,1.000000,NaN,0.694444,0.694444
1406,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,The Youth Risk Behavior Surveillance System (Y...,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,exact,100.000000,The Youth Risk Behavior Surveillance System (Y...,...,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000
1407,Youth Risk Behavior Surveillance System (YRBSS...,youth risk behavior surveillance system yrbss ...,youth risk behavior surveillance system yrbss,YRBSS monitors adolescent health behavior chan...,

In [70]:
drp_notes_lookup = drp.drop_duplicates("project_title").set_index("project_title")["notes"]
matches_checked["drp_notes"] = matches_checked["project_title"].map(drp_notes_lookup)

In [71]:
def get_best_abstract(row):
    dgm_exact = row.get('dgm_norm_match_type') == 'exact' and row.get('dgm_description', '') != ''
    dgf_exact = row.get('dgf_norm_match_type') == 'exact' and row.get('dgf_description', '') != ''

    if dgm_exact and dgf_exact:
        return row['dgf_description'], 'dgf_exact'
    if dgm_exact:
        return row['dgm_description'], 'dgm_exact'
    if dgf_exact:
        return row['dgf_description'], 'dgf_exact'

    # fuzzy: pick higher score, ties go to dgf
    dgm_score = row.get('dgm_norm_match_score') or 0
    dgf_score = row.get('dgf_norm_match_score') or 0
    if dgf_score >= dgm_score and row.get('dgf_description', '') != '':
        return row['dgf_description'], 'dgf_fuzzy'
    if row.get('dgm_description', '') != '':
        return row['dgm_description'], 'dgm_fuzzy'
    if row.get('dgf_description', '') != '':
        return row['dgf_description'], 'dgf_fuzzy'
    return '', 'none'


In [72]:
matches_checked[['best_abstract', 'best_abstract_source']] = matches_checked.apply(lambda r: pd.Series(get_best_abstract(r)), axis=1)

print(matches_checked['best_abstract_source'].value_counts().to_string())
print(f'Empty best_abstract: {(matches_checked["best_abstract"].fillna("") == "").sum():,}')
print(f'Total rows:          {len(matches_checked):,}')
matches_checked

best_abstract_source
dgf_exact    899
dgf_fuzzy    415
dgm_fuzzy     52
dgm_exact     44
Empty best_abstract: 11
Total rows:          1,410


,project_title,drp_title_norm,drp_title_topic,dc_abstract,dgm_title,dgm_title_norm,dgm_title_topic,dgm_norm_match_type,dgm_norm_match_score,dgm_description,...,dgf_topic_match_type,dgf_topic_match_score,sim_drp_dgm,sim_drp_dgf,sim_dgf_dgm,drp_notes,new_content_vs_dgm,new_content_vs_dgf,best_abstract,best_abstract_source
0,1998-2023 Serotype Data for Invasive Pneumococ...,1998 2023 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,CDC monitors invasive bacterial infections tha...,1998-2021 Serotype Data for Invasive Pneumococ...,1998 2021 serotype data for invasive pneumococ...,serotype data for invasive pneumococcal diseas...,fuzzy,99.137931,CDC monitors invasive bacterial infections tha...,...,exact,100.0,0.989752,0.988295,0.999354,NaN,0.150442,0.150442,CDC monitors invasive bacterial infections tha...,dgf_fuzzy
1,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,The map illustrates the total number of 2013 a...,2013-2014 PHAP Associates by State,2013 2014 phap associates by state,phap associates by state,exact,100.000000,The map illustrates the total number of 2013 a...,...,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000,The map illustrates the total number of 2013 a...,dgf_exact
2,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,This project includes a pdf capture of a webpa...,2014 Minority Veteran Report,2014 minority veteran report,minority veteran report,exact,100.000000,"Over the past 30 years, racial and ethnic mino...",...,exact,100.0,0.670759,0.670759,1.000000,NaN,0.875000,0.875000,"Over the past 30 years, racial and ethnic mino...",dgf_exact
3,2016 AmeriCorps MES AmeriCorps Member Exit Survey,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2016 AmeriCorps MES: AmeriCorps Member Exit Su...,2016 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,exact,100.0,0.971931,0.971931,1.000000,NaN,0.011628,0.011628,"Upon exiting service, AmeriCorps members are i...",dgf_exact
4,2017 AmeriCorps MES AmeriCorps Member Exit Survey,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,"Upon exiting service, AmeriCorps members are i...",2017 AmeriCorps MES: AmeriCorps Member Exit Su...,2017 americorps mes americorps member exit survey,americorps mes americorps member exit survey,exact,100.000000,"Upon exiting service, AmeriCorps members are i...",...,exact,100.0,0.972655,0.972655,1.000000,NaN,0.011628,0.011628,"Upon exiting service, AmeriCorps members are i...",dgf_exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,Women Veterans Forum,women veterans forum,women veterans forum,This project includes a pdf capture of a webpa...,Women Veterans Forum,women veterans forum,women veterans forum,exact,100.000000,"On February 7, 2017, the VA Office of Enterpri...",...,exact,100.0,0.851454,0.851454,1.000000,NaN,0.694444,0.694444,"On February 7, 2017, the VA Office of Enterpri...",dgf_exact
1406,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,The Youth Risk Behavior Surveillance System (Y...,YRBS State Tobacco Variables 2013 - v2,yrbs state tobacco variables 2013 v2,yrbs state tobacco variables,exact,100.000000,The Youth Risk Behavior Surveillance System (Y...,...,exact,100.0,1.000000,1.000000,1.000000,NaN,0.000000,0.000000,The Youth Risk Behavior Surveillance System (Y...,dgf_exact
1407,Youth Risk Behavior Surveillance System (YRBSS...,youth risk behavior surveillance system yrbss ...,youth risk behavior surveillance system yrbss,YRBSS monitors adolescent health behavior chan...,Youth Risk Behavior Surveillance System (YRBSS),youth risk behavior surveillance syste

In [75]:
dgf_matched_titles = set(matches_checked.loc[matches_checked['best_abstract_source'].str.startswith('dgf'),'dgf_title_norm'].dropna())
dgf_originals = (dgf[dgf['title_norm'].isin(dgf_matched_titles)].drop_duplicates(subset=['title', 'description']).reset_index(drop=True))

In [76]:
print(f'dgf titles used as best abstract: {len(dgf_matched_titles):,}')
print(f'dgf originals after dedup: {len(dgf_originals):,}')
dgf_originals

dgf titles used as best abstract: 1,182
dgf originals after dedup: 1,313


,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,dcat_landing_page,dcat_language,dcat_temporal,dcat_spatial,dcat_bureau_code,dcat_program_code,dcat_periodicity,dcat_license,dcat_rights,title_norm
0,U.S. Chronic Disease Indicators,CDC's Division of Population Health provides a...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/hksd-2xuw,u-s-chronic-disease-indicators,False,3700,2025-08-08T20:53:50.816467,"[alcohol, arthritis, asthma, cancer, copd, dia...",[Chronic Disease Indicators],...,https://www.cdc.gov/cdi/overview.html,[],,,[009:20],[009:020],,http://opendatacommons.org/licenses/odbl/1.0/,,us chronic disease indicators
1,"Nutrition, Physical Activity, and Obesity - Be...","This dataset includes data on adult's diet, ph...",Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/hn4x-zwk7,nutrition-physical-activity-and-obesity-behavi...,False,2498,2025-12-13T08:46:51.184555,"[adults, brfss, dnpao, fruit, nutrition, obesi...","[Nutrition, Physical Activity, and Obesity]",...,http://www.cdc.gov/nccdphp/DNPAO/index.html,[],,,[009:20],[009:020],,http://opendefinition.org/licenses/odc-odbl/,,nutrition physical activity and obesity behavi...
2,NCHS - Leading Causes of Death: United States,This dataset presents the age-adjusted death r...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/bi63-dtpu,nchs-leading-causes-of-death-united-states,True,1423,2025-08-09T02:10:37.492254,"[leading causes of death, mortality, nchs, sta...",[National Center for Health Statistics],...,https://www.cdc.gov/nchs/data-visualization/mo...,[],1999/2017,United States,[009:20],[009:020],,https://www.usa.gov/government-works,,nchs leading causes of death united states
3,Consumer Complaint Database,The Consumer Complaint Database is a collectio...,Consumer Financial Protection Bureau,CCDB,consumer-complaint-database,True,1282,2025-07-29T12:19:39.355656,"[bank account, bank service, complaint, consum...",[],...,https://www.consumerfinance.gov/data-research/...,[],,United States,[581:00],[000:000],R/P1D,,,consumer complaint database
4,Mental Health Care in the Last 4 Weeks,"The U.S. Census Bureau, in collaboration with ...",Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/yni7-er2q,mental-health-care-in-the-last-4-weeks,True,1232,2025-08-09T00:53:33.348252,"[covid-19, mental health]",[National Center for Health Statistics],...,https://www.cdc.gov/nchs/covid19/pulse/mental-...,[en-US],2020-08-19/2022-05-09,US,[009:20],[009:020],R/P2W,https://www.usa.gov/government-works,,mental health care in the last 4 weeks
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1308,NNDSS - TABLE 1KK. Vancomycin-intermediate Sta...,NNDSS - TABLE 1KK. Vancomycin-intermediate Sta...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/759d-qk63,nndss-table-1kk-vancomycin-intermediate-staphy...,False,0,2025-08-09T02:59:20.124993,"[2022, nedss, netss, nndss, vancomycin-interme...",[NNDSS],...,https://data.cdc.gov/d/759d-qk63,[],,,[009:20],[009:020],,http://opendefinition.org/licenses/odc-odbl/,,nndss table 1kk vancomycin intermediate staphy...
1309,Data describing highly pathogenic H5N1 in Doub...,These data describe the prevalence of HP H5N1 ...,U.S. Geological Survey,http://datainventory.doi.gov/id/dataset/USGS_6...,data-describing-highly-pathogenic-h5n1-in-doub...,True,0,2026-01-27T01:57:26.829044,"[USGS:665de174d34e19fd55a96bf6, avian influenz...",[Geospatial],...,,[],,"-76.39343, 38.74230, -76.35498, 38.79798",[010:12],[],,,,data describing highly pathogenic h5n1 in doub...
1310,Early Model-based Provisional Estimates of Dru...,This dataset provides model-based provisional ...,Centers for Disease Control and Prevention,https://data.cdc.gov/api/views/v2g4-wqg2,early-model-based-provisional-estimates-of-dru...,False,0,2025-08-09T06:01:09.679567,"[deaths, drug overdose, mo

In [ ]:
dgf_originals.to_parquet('00_data/02_03_drp_in_dgf_formodeling.parquet', index=False)


For appendix:
Entry manually selected using the Excel spreadsheet

In [78]:
example = 'FOIA Electronic Reading Room Data'
example_alt_title = 'FOIA Electronic Reading Room'

In [79]:
drp[drp['title'] == example]

,Unnamed: 0,file,schema,title,organization,agency,websites,data_source,last_modified,metadata_available,...,num_nyt_words_title,n_banned_terms_title,num_flagged_words_total,num_pen_words_total,num_nyt_words_total,n_banned_terms_total,has_flagged_word_total,title_norm,title_topic,description
634,634,_datasets/foia-electronic-reading-room-data.md,data_rescue_project,FOIA Electronic Reading Room Data,Transportation Security Administration,Department of Homeland Security,tsa.gov,https://www.tsa.gov/foia/readingroom,2025-05-23,True,...,0,0,0,0,0,0,0,foia electronic reading room data,foia electronic reading room data,This collection includes all PDFs available in...


In [80]:
dgf[dgf['title'] == example_alt_title]

,title,description,publisher,identifier,slug,has_spatial,popularity,last_harvested_date,keyword,theme,...,dcat_landing_page,dcat_language,dcat_temporal,dcat_spatial,dcat_bureau_code,dcat_program_code,dcat_periodicity,dcat_license,dcat_rights,title_norm
298189,FOIA Electronic Reading Room,The E-FOIA Amendments of 1996 require that fre...,Office of the Secretary,https://www.usitc.gov/secretary/foia/foia_erea...,foia-electronic-reading-room,True,0,2025-07-29T13:28:46.043957,"[FOIA, Freedom of Information Act, Reading Room]",[FOIA],...,https://www.usitc.gov/secretary/foia/foia_erea...,[en-US],2018-01-01/2018-09-28,United States,[378:00],[000:000],R/P1Y,http://www.usa.gov/publicdomain/label/1.0/,This dataset is a U.S. Government Work and is ...,foia electronic reading room


# Next: 03_deduplication